In [1]:
# %pip install pandas matplotlib seaborn
import requests
import zipfile
import io
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path



In [2]:
# Download Data 1 

# Create data directory
Path("data_test").mkdir(exist_ok=True)

# Download one month (October 2023)
url = "https://divvy-tripdata.s3.amazonaws.com/202310-divvy-tripdata.zip"
zip_path = "data_test/202310-divvy-tripdata.zip"

print("Downloading data...")
response = requests.get(url)
with open(zip_path, 'wb') as f:
    f.write(response.content)

# Extract
print("Extracting...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("data_test/")
    
print("Download complete!")

Extracting...
Download complete!


In [3]:
# Download Data 2

# Create a data directory
os.makedirs('data_test', exist_ok=True)

# Divvy data URL (October 2023)
url = "https://divvy-tripdata.s3.amazonaws.com/202411-divvy-tripdata.zip"
zip_name = "202411-divvy-tripdata.zip"

print("Downloading data...")
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

print("Extracting...")
z.extractall("data_test")
print("Done! Files in folder:", os.listdir("data_test"))

Extracting...
Done! Files in folder: ['202411-divvy-tripdata.csv', '202310-divvy-tripdata.zip', '__MACOSX', '202310-divvy-tripdata.csv']


In [5]:
import duckdb

con = duckdb.connect("data/warehouse.duckdb")

df = con.execute("""
    SELECT *
    FROM read_parquet('data/raw/*.parquet')
    LIMIT 10
""").df()

IOException: IO Error: Cannot open file "/workspaces/dezoomcamp-divvy-chicago-dbt/notebooks/data/warehouse.duckdb": No such file or directory

In [ ]:
from google.cloud import storage

def upload_to_gcs(bucket_name, source_file_name, destination_blob_name):
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(source_file_name)
    print(f"File {source_file_name} uploaded to {destination_blob_name}.")

# upload_to_gcs('your-bucket-name', 'data/202310-divvy-tripdata.csv', 'raw/202310.csv')

In [ ]:
# Load and Inspect
print(list(Path("data").glob("*.csv")))



In [ ]:
# Find the CSV file
csv_file = list(Path("data").glob("*.csv"))[0]
print(f"Loading: {csv_file}")

# Load data
df = pd.read_csv(csv_file)

print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Basic Info
df.info()



In [ ]:
# Summary Statistics
df.describe()

In [ ]:
# Data Quality Checks
print("Missing values:")
print(df.isnull().sum())

print("\n" + "="*50)
print("Unique ride types:")
print(df['rideable_type'].value_counts())

print("\n" + "="*50)
print("Member vs Casual:")
print(df['member_casual'].value_counts())

In [ ]:
# DateTime Analysis
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])

# Calculate trip duration
df['trip_duration_min'] = (df['ended_at'] - df['started_at']).dt.total_seconds() / 60

# Add time features
df['hour'] = df['started_at'].dt.hour
df['day_of_week'] = df['started_at'].dt.day_name()

print("Trip duration stats (minutes):")
print(df['trip_duration_min'].describe())



In [ ]:
# Visualizations
plt.style.use('seaborn-v0_8-darkgrid')
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Trips by hour
df['hour'].value_counts().sort_index().plot(kind='bar', ax=axes[0,0])
axes[0,0].set_title('Trips by Hour of Day')
axes[0,0].set_xlabel('Hour')

# Trips by day of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['day_of_week'].value_counts()[day_order].plot(kind='bar', ax=axes[0,1])
axes[0,1].set_title('Trips by Day of Week')

# Trip duration distribution (filtered outliers)
df[df['trip_duration_min'] < 60]['trip_duration_min'].hist(bins=50, ax=axes[1,0])
axes[1,0].set_title('Trip Duration Distribution (< 60 min)')
axes[1,0].set_xlabel('Minutes')

# Member vs Casual
df['member_casual'].value_counts().plot(kind='pie', ax=axes[1,1], autopct='%1.1f%%')
axes[1,1].set_title('Member vs Casual Riders')

plt.tight_layout()
plt.show()

In [ ]:
# Station Analysis
print("Top 10 Start Stations:")
print(df['start_station_name'].value_counts().head(10))

print("\n" + "="*50)
print("Top 10 End Stations:")
print(df['end_station_name'].value_counts().head(10))

In [ ]:
# Data Quality Issues to Address
print("Data Quality Issues Found:")
print(f"1. Missing station names: {df['start_station_name'].isnull().sum()}")
print(f"2. Negative trip durations: {(df['trip_duration_min'] < 0).sum()}")
print(f"3. Very long trips (>24h): {(df['trip_duration_min'] > 1440).sum()}")
print(f"4. Very short trips (<1min): {(df['trip_duration_min'] < 1).sum()}")